# ScienceQA Visual Challenge — Optimized Notebook

**Strategy summary:**
- **Model**: `HuggingFaceTB/SmolVLM-500M-Instruct` (required by rules)
- **Fine-tuning**: QLoRA (4-bit NF4), LoRA r=16 on all attention projections — verified ≤5M trainable params
- **Inference**: Log-likelihood scoring over each choice letter (replaces unreliable generation)
- **Prompts**: lecture + hint + solution (chain-of-thought) during training
- **Augmentation**: Random choice-order shuffling to prevent positional bias
- **Final model**: Retrained on train+val combined before submission
- **Ensemble**: 3 seeds, average log-probs (optional section at bottom)

In [1]:
# ── 0. Install ────────────────────────────────────────────────────────────────
# transformers 4.47.1: Idefics3Processor works correctly, no vmap masking bug.
# The vmap-based masking_utils that crash with bitsandbytes 4-bit landed in 4.48.
!pip install -q transformers==4.47.1 peft==0.14.0 bitsandbytes accelerate datasets pillow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.8/374.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 80.6 MB/s eta 0:00:00


In [2]:
# ── 1. Imports & Configuration ────────────────────────────────────────────────
import os
import gc
import json
import random
from pathlib import Path
from functools import partial

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from transformers import (
    Idefics3Processor,
    Idefics3ForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/input/datasets/terrykom/finalexamdataset")

# ── Model & training knobs ────────────────────────────────────────────────────
MODEL_ID       = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE       = 224
LORA_R         = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05
LR             = 1e-4
NUM_EPOCHS     = 2          # 2 epochs × ~3.8 h/epoch ≈ 7.6 h (safely under 10 h)
BATCH_SIZE     = 2
GRAD_ACCUM     = 8
CHOICE_LETTERS = "ABCDEFGHIJ"
PARAM_BUDGET   = 5_000_000

# ── Text-field limits ─────────────────────────────────────────────────────────
INCLUDE_LECTURE    = False
SOLUTION_MAX_CHARS = 400

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

2026-04-30 20:51:29.528970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777582289.778540      22 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777582289.844746      22 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777582290.410945      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777582290.410984      22 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777582290.410987      22 computation_placer.cc:177] computation placer alr

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 2. Load Data

In [3]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

# Detect whether solution is available at test time.
# In ScienceQA the test split typically omits solutions since they reveal the answer.
# We check dynamically so the code works either way.
TEST_HAS_SOLUTION = (
    "solution" in test_df.columns
    and test_df["solution"].notna().any()
    and test_df["solution"].astype(str).str.strip().ne("").any()
)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
print(f"Solution available in test.csv: {TEST_HAS_SOLUTION}")
print(f"Test columns: {test_df.columns.tolist()}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008
Solution available in test.csv: False
Test columns: ['id', 'image_path', 'question', 'choices', 'num_choices', 'hint', 'lecture', 'task', 'grade', 'subject', 'topic', 'category', 'skill']


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [4]:
# ── 2b. Prompt Engineering ────────────────────────────────────────────────────

def _trunc(text, max_chars: int) -> str:
    text = str(text).strip()
    return text if len(text) <= max_chars else text[:max_chars] + "…"


def build_prompt(
    row,
    include_answer: bool = False,
    include_solution: bool = False,
) -> str:
    context_parts = []

    # lecture excluded by default (INCLUDE_LECTURE=False) — saves ~125 tokens/sample
    if INCLUDE_LECTURE:
        lecture = row.get("lecture", "") if isinstance(row, dict) else getattr(row, "lecture", "")
        if pd.notna(lecture) and str(lecture).strip():
            context_parts.append(_trunc(lecture, 500))

    hint = row.get("hint", "") if isinstance(row, dict) else getattr(row, "hint", "")
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())

    if include_solution:
        sol = row.get("solution", "") if isinstance(row, dict) else getattr(row, "solution", "")
        if pd.notna(sol) and str(sol).strip():
            context_parts.append("Reasoning: " + _trunc(sol, SOLUTION_MAX_CHARS))

    choices = row["choices"] if isinstance(row, dict) else row.choices
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_parts:
        prompt += "Context:\n" + "\n\n".join(context_parts) + "\n\n"
    question = row["question"] if isinstance(row, dict) else row.question
    prompt += f"Question: {question}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer:"

    if include_answer:
        answer = row["answer"] if isinstance(row, dict) else row.answer
        prompt += f" {CHOICE_LETTERS[int(answer)]}"

    return prompt

sample = build_prompt(train_df.iloc[0].to_dict(), include_answer=True, include_solution=True)
print(f"Sample prompt length: {len(sample)} chars")
print(sample[:300], "…")

Sample prompt length: 1845 chars
<image>
Context:
Animals often behave in certain ways that can increase their reproductive success. Read the passage about a specific animal behavior. Then, follow the instructions below.

Amazonian poison frogs live in tropical forests in northern South America. After a male and female frog mate, t …


In [5]:
# ── 2c. Dataset with choice-order augmentation ────────────────────────────────

class ScienceQADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        data_dir: Path,
        img_size: int = IMG_SIZE,
        is_train: bool = True,
        augment_choices: bool = False,
        include_solution: bool = False,
    ):
        self.df               = df.reset_index(drop=True)
        self.data_dir         = data_dir
        self.img_size         = img_size
        self.is_train         = is_train
        self.augment_choices  = augment_choices
        self.include_solution = include_solution

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx].to_dict()

        img = Image.open(self.data_dir / "images" / row["image_path"]).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)

        if self.augment_choices and self.is_train:
            choices = list(row["choices"])
            answer  = int(row["answer"])
            perm    = list(range(len(choices)))
            random.shuffle(perm)
            row["choices"] = [choices[p] for p in perm]
            row["answer"]  = perm.index(answer)

        if self.is_train:
            return {
                "image": img,
                "text":  build_prompt(row, include_answer=True,
                                      include_solution=self.include_solution),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False,
                                        include_solution=self.include_solution),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row and pd.notna(row.get("answer")) else -1,
                "id":      row.get("id", ""),
            }

# include_solution mirrors TEST_HAS_SOLUTION so training and inference see
# identical prompt formats — mismatch here was the primary cause of low accuracy.
train_ds = ScienceQADataset(train_df, DATA_DIR, is_train=True,
                            augment_choices=True,
                            include_solution=TEST_HAS_SOLUTION)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, is_train=False,
                            include_solution=TEST_HAS_SOLUTION)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, is_train=False,
                            include_solution=TEST_HAS_SOLUTION)

print(f"Datasets — train: {len(train_ds)} | val: {len(val_ds)} | test: {len(test_ds)}")
print(f"include_solution in all datasets: {TEST_HAS_SOLUTION}")

Datasets — train: 3109 | val: 1048 | test: 1008
include_solution in all datasets: False


## 3. Load Model with QLoRA

The parameter budget is **5 million trainable params**. With `r=16` on the four attention projections (`q, k, v, o`) of SmolVLM-500M's language backbone this lands at roughly 2–4 M params — safely within budget. `model.print_trainable_parameters()` shows the exact count. If it exceeds 5 M, change `LORA_R = 8`.

In [6]:
# ── 3. Load SmolVLM-500M with 4-bit QLoRA ────────────────────────────────────

def load_model_and_processor(model_id: str = MODEL_ID, lora_r: int = LORA_R):
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    # Use Idefics3Processor directly — AutoProcessor's auto-mapping for this
    # class is missing in transformers 4.47.x, causing it to return GPT2TokenizerFast.
    proc = Idefics3Processor.from_pretrained(model_id)
    if proc.tokenizer.pad_token is None:
        proc.tokenizer.pad_token = proc.tokenizer.eos_token
    proc.tokenizer.padding_side = "right"

    base = Idefics3ForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
    )
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=LORA_ALPHA,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
    )
    mdl = get_peft_model(base, lora_cfg)
    mdl.enable_input_require_grads()

    trainable, total = mdl.get_nb_trainable_parameters()
    print(f"Trainable params : {trainable:,} / {total:,}  ({100 * trainable / total:.3f}%)")
    assert trainable <= PARAM_BUDGET, (
        f"Trainable params {trainable:,} exceed budget {PARAM_BUDGET:,}. "
        "Reduce LORA_R and re-run."
    )
    print(f"padding_side     : {proc.tokenizer.padding_side}")
    print(f"pad_token_id     : {proc.tokenizer.pad_token_id}  ({proc.tokenizer.pad_token!r})")
    print(f"eos_token_id     : {proc.tokenizer.eos_token_id}  ({proc.tokenizer.eos_token!r})")
    return mdl, proc


model, processor = load_model_and_processor()

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Trainable params : 4,784,128 / 512,266,432  (0.934%)
padding_side     : right
pad_token_id     : 2  ('<|im_end|>')
eos_token_id     : 49279  ('<end_of_utterance>')


In [7]:
# ── 4. Collate function — answer-token label masking ─────────────────────────
#
# Training text always ends: "…Answer: X"  (X = one letter A–J).
# The processor may append one EOS token after X, and pads shorter sequences
# on the right (padding_side="right" is forced in load_model_and_processor).
#
# Strategy: start at the LAST position in the token sequence and walk backward
# past any EOS / padding tokens. The first token we hit that is neither EOS
# nor PAD is the answer letter. Unmask exactly that one position.
#
# Using len(seq)-1 (last index) rather than real_end-1 makes this robust to
# both right-padding AND left-padding (the Idefics3 default for generation).

def collate_fn(batch, proc):
    texts  = [item["text"]  for item in batch]
    images = [item["image"] for item in batch]

    # Defensive: ensure right-padding each call so attention_mask layout is
    # predictable even if something outside changes padding_side.
    proc.tokenizer.padding_side = "right"

    inputs = proc(text=texts, images=images, return_tensors="pt", padding=True)

    eos_id = proc.tokenizer.eos_token_id
    pad_id = proc.tokenizer.pad_token_id

    labels = torch.full_like(inputs["input_ids"], -100)

    for i in range(labels.shape[0]):
        seq = inputs["input_ids"][i].tolist()

        # Walk backward from the very last token, skipping EOS and PAD.
        answer_pos = len(seq) - 1
        while answer_pos >= 0 and seq[answer_pos] in (eos_id, pad_id):
            answer_pos -= 1

        if answer_pos >= 0:
            labels[i, answer_pos] = inputs["input_ids"][i, answer_pos]
        # If answer_pos < 0, no label is set — this will surface as NaN loss
        # and the sanity-check cell will catch it before training starts.

    inputs["labels"] = labels
    return inputs


_collate = partial(collate_fn, proc=processor)

In [8]:
# ── 4b. Pre-training sanity checks ───────────────────────────────────────────
# Runs BEFORE training. All checks must pass (no "FAIL" lines) before proceeding.
#
# CHECK A: collate_fn label positions — verifies the answer token is found.
# CHECK B: forward-pass loss — verifies the initial loss is finite and >0.
#
# If CHECK A fails: the collate_fn cannot find the answer token (padding bug).
# If CHECK B fails: labels are all -100 (loss=NaN) or model is broken.

print(f"{'='*65}")
print(f" Tokenizer  padding_side : {processor.tokenizer.padding_side}")
print(f"            pad_token_id : {processor.tokenizer.pad_token_id}")
print(f"            eos_token_id : {processor.tokenizer.eos_token_id}")
print(f"{'='*65}\n")

# ── CHECK A: label positions ──────────────────────────────────────────────────
print("── CHECK A: label positions ─────────────────────────────────────────")
_items = [train_ds[i] for i in range(4)]
_batch = _collate(_items)

eos_id = processor.tokenizer.eos_token_id
pad_id = processor.tokenizer.pad_token_id
check_a_ok = True

for i, item in enumerate(_items):
    seq    = _batch["input_ids"][i].tolist()
    labels = _batch["labels"][i].tolist()
    labeled = [p for p, l in enumerate(labels) if l != -100]
    expected = CHOICE_LETTERS[item["answer"]]

    print(f"  Sample {i}  (expected: {expected})")
    print(f"    seq length     : {len(seq)}")
    print(f"    labeled pos(s) : {labeled}")

    # Last 6 real tokens
    last_real = [p for p, m in enumerate(_batch["attention_mask"][i].tolist()) if m == 1][-6:]
    tokens_str = [(processor.tokenizer.decode([seq[p]]), seq[p]) for p in last_real]
    print(f"    last 6 tokens  : {tokens_str}")

    if len(labeled) == 0:
        print(f"    FAIL: no labeled position — collate_fn did not find the answer token")
        check_a_ok = False
    elif len(labeled) > 1:
        print(f"    FAIL: {len(labeled)} labeled positions (expected 1)")
        check_a_ok = False
    else:
        pos    = labeled[0]
        tok_id = seq[pos]
        tok_str = processor.tokenizer.decode([tok_id]).strip()
        if tok_id in (eos_id, pad_id):
            print(f"    FAIL: labeled token id={tok_id} is EOS or PAD, not a letter")
            check_a_ok = False
        elif tok_str.upper() != expected:
            print(f"    FAIL: decoded '{tok_str}' != expected '{expected}'")
            check_a_ok = False
        else:
            print(f"    OK  : token '{tok_str}' at pos {pos} matches '{expected}'")
    print()

# ── CHECK B: forward-pass loss ────────────────────────────────────────────────
print("── CHECK B: forward-pass loss ───────────────────────────────────────")
model.eval()
with torch.no_grad():
    _dev_batch = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in _batch.items()}
    out = model(**_dev_batch)
    loss_val = out.loss.item()

if loss_val != loss_val:  # NaN check
    print(f"  FAIL: loss = NaN — labels are probably all -100 (collate_fn bug)")
    check_b_ok = False
elif loss_val < 0.01:
    print(f"  FAIL: loss = {loss_val:.4f} — suspiciously near zero before any training")
    check_b_ok = False
elif loss_val > 20:
    print(f"  WARN: loss = {loss_val:.4f} — unusually high (check inputs)")
    check_b_ok = True
else:
    print(f"  OK  : initial loss = {loss_val:.4f}  (expected ~2–5 for a random model)")
    check_b_ok = True
model.train()

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n{'='*65}")
if check_a_ok and check_b_ok:
    print("  ALL CHECKS PASSED — safe to proceed to training")
else:
    print("  ONE OR MORE CHECKS FAILED — DO NOT PROCEED until fixed")
print(f"{'='*65}")

 Tokenizer  padding_side : right
            pad_token_id : 2
            eos_token_id : 49279

── CHECK A: label positions ─────────────────────────────────────────
  Sample 0  (expected: C)
    seq length     : 1438
    labeled pos(s) : [1437]
    last 6 tokens  : [(' adult', 2490), (' frogs', 18952), ('\n', 198), ('Answer', 21350), (':', 42), (' C', 340)]
    OK  : token 'C' at pos 1437 matches 'C'

  Sample 1  (expected: C)
    seq length     : 1438
    labeled pos(s) : [1411]
    last 6 tokens  : [(' live', 2330), (' longer', 2848), ('\n', 198), ('Answer', 21350), (':', 42), (' C', 340)]
    OK  : token 'C' at pos 1411 matches 'C'

  Sample 2  (expected: A)
    seq length     : 1438
    labeled pos(s) : [1410]
    last 6 tokens  : [(' other', 550), (' cubs', 40684), ('\n', 198), ('Answer', 21350), (':', 42), (' A', 330)]
    OK  : token 'A' at pos 1410 matches 'A'

  Sample 3  (expected: A)
    seq length     : 1438
    labeled pos(s) : [1415]
    last 6 tokens  : [(' get', 820), 

## 4. Training — Development Run

Train on `train_df`, evaluate on `val_df`. Use this to confirm the model learns and to pick the best epoch checkpoint. The `load_best_model_at_end=True` flag restores the lowest-val-loss checkpoint automatically.

In [9]:
# ── 4c. Smoke test — 50 samples × 1 epoch (~5 min on T4) ─────────────────────
from transformers import TrainerCallback

class VerboseCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or "loss" not in logs:
            return
        loss  = logs["loss"]
        step  = state.global_step
        epoch = logs.get("epoch", "?")
        lr    = logs.get("learning_rate", float("nan"))
        if loss != loss:
            print(f"  Step {step:4d} | epoch {epoch} | loss = NaN  ← labels may all be -100")
        elif loss < 1e-4:
            print(f"  Step {step:4d} | epoch {epoch} | loss = {loss:.6f}  ← suspiciously low")
        else:
            print(f"  Step {step:4d} | epoch {epoch} | loss = {loss:.4f} | lr = {lr:.2e}")


smoke_ds = ScienceQADataset(
    train_df.head(50), DATA_DIR,
    is_train=True, augment_choices=False,
    include_solution=TEST_HAS_SOLUTION,
)

smoke_args = TrainingArguments(
    output_dir           = "./smoke-test",
    num_train_epochs     = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 2,
    learning_rate        = LR,
    lr_scheduler_type    = "cosine",
    warmup_ratio         = 0.1,
    fp16                 = True,
    bf16                 = False,
    gradient_checkpointing = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    save_strategy        = "no",
    eval_strategy        = "no",
    logging_steps        = 5,
    dataloader_num_workers = 0,
    remove_unused_columns  = False,
    report_to            = "none",
)

model.config.use_cache = False

smoke_trainer = Trainer(
    model        = model,
    args         = smoke_args,
    train_dataset= smoke_ds,
    data_collator= _collate,
    callbacks    = [VerboseCallback()],
)

print("=== SMOKE TEST START (50 samples, 1 epoch) ===")
smoke_trainer.train()
print("=== SMOKE TEST DONE ===")

=== SMOKE TEST START (50 samples, 1 epoch) ===


Step,Training Loss
5,5.481800
10,3.155200


  Step    5 | epoch 0.4 | loss = 5.4818 | lr = 1.00e-04
  Step   10 | epoch 0.8 | loss = 3.1552 | lr = 5.00e-05
=== SMOKE TEST DONE ===


In [10]:
# # ── 5. Dev training run ───────────────────────────────────────────────────────
# del model
# gc.collect()
# torch.cuda.empty_cache()

# model, processor = load_model_and_processor()
# _collate = partial(collate_fn, proc=processor)

# training_args_dev = TrainingArguments(
#     output_dir                  = "./qlora-dev",
#     num_train_epochs            = NUM_EPOCHS,
#     per_device_train_batch_size = BATCH_SIZE,
#     gradient_accumulation_steps = GRAD_ACCUM,
#     learning_rate               = LR,
#     lr_scheduler_type           = "cosine",
#     warmup_ratio                = 0.1,
#     fp16                        = True,
#     bf16                        = False,
#     gradient_checkpointing      = True,
#     gradient_checkpointing_kwargs = {"use_reentrant": False},
#     save_strategy               = "no",
#     eval_strategy               = "no",
#     logging_steps               = 10,
#     dataloader_num_workers      = 0,
#     remove_unused_columns       = False,
#     report_to                   = "none",
# )

# model.config.use_cache = False

# trainer_dev = Trainer(
#     model         = model,
#     args          = training_args_dev,
#     train_dataset = train_ds,
#     data_collator = _collate,
#     callbacks     = [VerboseCallback()],
# )

# print("=== DEV TRAINING START ===")
# trainer_dev.train()
# print("=== DEV TRAINING DONE ===")

In [11]:
# # ── 5b. Save dev model (run immediately after training to avoid losing progress)
# # Saves only the LoRA adapter weights (~20 MB), not the frozen base.
# # Reload later with: model = PeftModel.from_pretrained(base, "./lora-dev-seed42")

# model.save_pretrained("./lora-dev-seed42")
# processor.save_pretrained("./lora-dev-seed42")
# print("Dev adapter saved to ./lora-dev-seed42")

## 6. Final Training on train + val

With only ~4,157 labeled examples total, every sample matters. We reload a fresh base model and retrain on the full labeled set (train + val) for the same number of epochs as the best dev run. No early stopping — we use `NUM_EPOCHS` directly.

In [12]:
# ── 7. Final training (train + val) ──────────────────────────────────────────
del model
gc.collect()
torch.cuda.empty_cache()

model, processor = load_model_and_processor()
_collate = partial(collate_fn, proc=processor)

full_df       = pd.concat([train_df, val_df], ignore_index=True)
full_train_ds = ScienceQADataset(
    full_df, DATA_DIR,
    is_train=True,
    augment_choices=True,
    include_solution=TEST_HAS_SOLUTION,
)

training_args_final = TrainingArguments(
    output_dir                    = "./qlora-final",
    num_train_epochs              = NUM_EPOCHS,
    per_device_train_batch_size   = BATCH_SIZE,
    gradient_accumulation_steps   = GRAD_ACCUM,
    learning_rate                 = LR,
    lr_scheduler_type             = "cosine",
    warmup_ratio                  = 0.1,
    fp16                          = True,
    bf16                          = False,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {"use_reentrant": False},
    # ── Checkpoint every 50 optimizer steps (~44 min) so a crash can be resumed.
    # To resume: trainer_final.train(resume_from_checkpoint="./qlora-final/checkpoint-XXXX")
    # or set resume_from_checkpoint=True to pick the latest checkpoint automatically.
    save_strategy                 = "steps",
    save_steps                    = 50,
    save_total_limit              = 3,      # keeps last 3 checkpoints (~60 MB each)
    eval_strategy                 = "no",
    logging_steps                 = 10,
    dataloader_num_workers        = 0,
    remove_unused_columns         = False,
    report_to                     = "none",
)

model.config.use_cache = False

trainer_final = Trainer(
    model         = model,
    args          = training_args_final,
    train_dataset = full_train_ds,
    data_collator = _collate,
    callbacks     = [VerboseCallback()],
)

print("=== FINAL TRAINING START ===")
# To resume from a crash: pass resume_from_checkpoint=True (latest) or a path
# trainer_final.train(resume_from_checkpoint=True)
trainer_final.train()
print("=== FINAL TRAINING DONE ===")

model.save_pretrained("./lora-final-seed42")
processor.save_pretrained("./lora-final-seed42")
print("Adapter saved to ./lora-final-seed42")

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Trainable params : 4,784,128 / 512,266,432  (0.934%)
padding_side     : right
pad_token_id     : 2  ('<|im_end|>')
eos_token_id     : 49279  ('<end_of_utterance>')
=== FINAL TRAINING START ===


Step,Training Loss
10,14.420000
20,13.038700
30,11.028400
40,9.046800
50,7.629500
60,7.820500
70,6.179900
80,6.768600
90,5.067800
100,6.252900


  Step   10 | epoch 0.03848003848003848 | loss = 14.4200 | lr = 1.15e-05
  Step   20 | epoch 0.07696007696007696 | loss = 13.0387 | lr = 2.88e-05
  Step   30 | epoch 0.11544011544011544 | loss = 11.0284 | lr = 4.81e-05
  Step   40 | epoch 0.15392015392015393 | loss = 9.0468 | lr = 6.73e-05
  Step   50 | epoch 0.1924001924001924 | loss = 7.6295 | lr = 8.65e-05
  Step   60 | epoch 0.23088023088023088 | loss = 7.8205 | lr = 1.00e-04
  Step   70 | epoch 0.26936026936026936 | loss = 6.1799 | lr = 9.98e-05
  Step   80 | epoch 0.30784030784030786 | loss = 6.7686 | lr = 9.94e-05
  Step   90 | epoch 0.3463203463203463 | loss = 5.0678 | lr = 9.88e-05
  Step  100 | epoch 0.3848003848003848 | loss = 6.2529 | lr = 9.79e-05
  Step  110 | epoch 0.42328042328042326 | loss = 5.2450 | lr = 9.68e-05
  Step  120 | epoch 0.46176046176046176 | loss = 5.5412 | lr = 9.56e-05
  Step  130 | epoch 0.5002405002405003 | loss = 5.3495 | lr = 9.41e-05
  Step  140 | epoch 0.5387205387205387 | loss = 4.6011 | lr = 9.2